In [19]:
import pandas as pd
import numpy as np
from openai import OpenAI
import os
from typing import List, Tuple 
from langchain.evaluation import load_evaluator
from typing import List, Dict, Any

import pandas as pd
from dotenv import load_dotenv

In [20]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY")
) 
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [21]:
def preprocess_text(text: str) -> str:
    if pd.isna(text):
        return ""
    return str(text).strip()

def create_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> Tuple[List[List[float]], int]:
    embeddings = []
    for i in range(0, len(texts), 100):
        batch = texts[i:i+100]
        try:
            # Create embeddings
            response = client.embeddings.create(
                input=batch,
                model=model
            )
            # Extract embeddings from the response
            batch_embeddings = [item.embedding for item in response.data]
            embeddings.extend(batch_embeddings)
            
            # Get embedding dimension from first embedding
            embedding_dim = len(batch_embeddings[0]) if batch_embeddings else 0
        except Exception as e:
            print(f"Error creating embeddings for batch {i}: {e}")
            # Add placeholder embedding if error occurs
            placeholder_dim = 1536  # Default embedding dimension
            embeddings.extend([np.zeros(placeholder_dim).tolist()] * len(batch))
            embedding_dim = placeholder_dim
    
    return embeddings, embedding_dim



In [22]:
def vectorize_csv_sheet(filepath: str, columns_to_embed: List[str]) -> pd.DataFrame:

    # Read the Excel file
    df = pd.read_csv(filepath)    
    # Preprocess and combine text from specified columns
    df['combined_text'] = df[columns_to_embed].apply(lambda row: ' '.join(
        preprocess_text(str(val)) for val in row
    ), axis=1)
    
    # Create embeddings
    embeddings, embedding_dim = create_embeddings(df['combined_text'].tolist())
    
    # Create embedding columns efficiently using NumPy and pd.concat
    embedding_cols = pd.DataFrame(
        embeddings, 
        columns=[f'embedding_{i}' for i in range(embedding_dim)]
    )
    
    # Combine original DataFrame with embedding columns
    return pd.concat([df, embedding_cols], axis=1)

In [23]:
def vectorize():
    # Example usage
    filepath = 'data/poverty.csv'
    columns_to_embed = ['Georgia!!Below poverty level!!Estimate', 'Georgia!!Below poverty level!!Margin of Error', "Georgia!!Below poverty level!!Estimate", "Georgia!!Below poverty level!!Margin of Error","Georgia!!Percent below poverty level!!Estimate","Georgia!!Percent below poverty level!!Margin of Error"]
    
    # Vectorize the Excel sheet
    vectorized_df = vectorize_csv_sheet(filepath, columns_to_embed)
    # print(vectorized_df)
    # Save the vectorized DataFrame
    vectorized_df.to_csv('vectorized_output.csv', index=False)
   
    print("Embedding process completed!")

In [24]:
vectorize()

Embedding process completed!


In [25]:
class RAGAssistant:
    def __init__(self, embeddings_file: str):
        """
        Initialize RAG Assistant with embedded data
        
        Args:
            embeddings_file: Path to Excel file with embeddings
        """
        # Load the embeddings file
        self.df = pd.read_csv(embeddings_file)
        
        # Get embedding column names
        self.embedding_cols = [col for col in self.df.columns if col.startswith('embedding_')]
        
        # Store original columns for context retrieval
        self.context_columns = [col for col in self.df.columns 
                                if col not in ['combined_text'] + self.embedding_cols]

    def calculate_embedding_similarity(self, query_embedding: List[float], doc_embedding: List[float]) -> float:
        """
        Calculate cosine similarity between query and document embeddings
        
        Args:
            query_embedding: Embedding of the query
            doc_embedding: Embedding of a document
        
        Returns:
            Cosine similarity score
        """
        return np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
        )

    def embed_query(self, query: str, model: str = "text-embedding-3-small") -> List[float]:
        """
        Generate embedding for a given query
        
        Args:
            query: Input query to embed
            model: OpenAI embedding model
        
        Returns:
            Embedding vector for the query
        """
        response = client.embeddings.create(
            input=[query],
            model=model
        )
        return response.data[0].embedding

    def retrieve_top_k_documents(self, query: str, k: int = 3) -> List[Dict[str, Any]]:
        """
        Retrieve top K most similar documents based on embedding similarity
        
        Args:
            query: User's query
            k: Number of top documents to retrieve
        
        Returns:
            List of top K documents with context
        """
        # Embed the query
        query_embedding = self.embed_query(query)
        
        # Calculate similarities
        self.df['similarity'] = self.df[self.embedding_cols].apply(
            lambda row: self.calculate_embedding_similarity(
                query_embedding, 
                row[self.embedding_cols]
            ), 
            axis=1
        )
        
        # Sort and get top K documents
        top_k_docs = self.df.sort_values('similarity', ascending=False).head(k)
        
        # Prepare context documents
        context_docs = []
        for _, row in top_k_docs.iterrows():
            context = {
                'similarity': row['similarity'],
                'context': {col: row[col] for col in self.context_columns}
            }
            context_docs.append(context)
        
        return context_docs

    def generate_response(self, query: str, model: str = "gpt-4o-mini") -> str:
        """
        Generate a response using RAG approach
        
        Args:
            query: User's input query
            model: OpenAI language model to use
        
        Returns:
            Generated response
        """
        # Retrieve top relevant documents
        retrieved_docs = self.retrieve_top_k_documents(query)
        
        # Prepare context for the prompt
        context = "\n\n".join([
            f"Document {i+1} (Similarity: {doc['similarity']:.2f}):\n" + 
            "\n".join(f"{k}: {v}" for k, v in doc['context'].items())
            for i, doc in enumerate(retrieved_docs)
        ])
        
        # Construct the prompt with retrieved context
        messages = [
            {
                "role": "system", 
                "content": "You are a helpful assistant that answers questions using the provided context. "
                           "If the context doesn't contain enough information, say so."
            },
            {
                "role": "user", 
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ]
        
        # Generate response
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=300
        )
        
        return response.choices[0].message.content


In [31]:
def main():
    # Example usage
    embeddings_file = 'vectorized_output.csv'
    rag_assistant = RAGAssistant(embeddings_file)
    
    # Example queries
    queries = [
        "What is the estimated population of people in poverty"
    ]
    
    # Generate responses
    for query in queries:
        print("-" * 50)
        print(f"Query: {query}")
        response = rag_assistant.generate_response(query)
        print("Response:", response)



In [32]:
main()

--------------------------------------------------
Query: What is the estimated population of people in poverty
Response: To find the estimated population of people in poverty in Georgia, we can look at the estimates provided for the different groups:

1. From Document 1: 96,300 people below the poverty level (Bachelor's degree or higher).
2. From Document 2: 1,011,497 people below the poverty level (Population 16 years and over).
3. From Document 3: 353,039 people below the poverty level (5 to 17 years).

The estimated populations in poverty are:
- Bachelor's degree or higher: 96,300
- Population 16 years and over: 1,011,497
- Ages 5 to 17: 353,039

However, since these estimates are possibly overlapping (for example, some individuals may belong to more than one group), we cannot simply sum them to get a total estimated population in poverty without more information on the overlaps.

Based on Document 2, the estimated people below the poverty level for the total population (16 years a